## Running Accelerator Toolbox in a simplified view.

*Sverker Werin 2026*

IPAC 2026 version


In the Load and initialize section **Accelerator Toolbox** is installed the first time you run the script (once per session).

1. Start by running the complete Notebook (Windows: ctrl-F9 or from the menu: >Run all or >Run>Run all. On a mobile the menu may be hidden behind a "down arrow")
2. Open a module by pressing the ">" beside the title or press "cells hidden" (if you open the code, double-click the title to close)
3. Calculations are made by push-buttons
4. To edit or add a new lattice you need to open the "User lattices" cell in the "Latticemodule" (double-click the title. Double-click the title to close)
5. Running a cell is normally not necessary (Run a cell by pressing the arrow at the title or press shift-enter)

More instructions on: https://github.com/werin99/pyAT_interface


A template lattice is prepared by default "MAX I 550 MeV (template)". This is one cell of the MAX I storage ring.


In [ ]:
#@title #Safe initialisation

# ==========================================
# 1. Initialization & Library Imports
# ==========================================

import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display, clear_output
try:
    import at
except ImportError:
    print("Installing accelerator-toolbox...")
    import os
    os.system('pip install "accelerator-toolbox>=0.3.0"')
    import at

# ==========================================
# 2. Core Classes & Data Containers
# ==========================================

class LatticeClass:
    """Template configuration structure for storage rings."""
    name = 'myLattice'
    elements = pd.DataFrame(
        {
            1: ['Drift_empty', 0.0, 0, 0],
            2: ['Drift_empty', 0.0, 0, 0]
        },
        index=['type', 'length', 'bending angle', 'K-value']
    )

class LatticeData:
    """Container to store computed optical parameters."""
    def __init__(self):
        self.s = None
        self.beta = None
        self.disp_fu = None
        self.alpha = None
        self.gamma = None
        self.mu = None

# Global tracking data instance
data = LatticeData()

# ==========================================
# 3. Tunespace functions
# ==========================================

def tunespace(tune, x_in, y_in, order):
    """Plots the betatron tune space and resonance lines up to 4th order."""
    plt.scatter(tune[0], tune[1], marker='o', s=100)

    # Calculate coordinate offsets and bounds
    x_off, y_off = x_in[0], y_in[0]
    x_min, x_max = x_in[0] - x_off, x_in[1] - x_off
    y_min, y_max = y_in[0] - y_off, y_in[1] - y_off

    f_0 = np.maximum(x_max - x_min, y_max - y_min)

    if order > 4:
        print('Maximum order supported is 4')
        return

    # Line styles mapped to resonance order
    line_styles = ['-b', '--r', '-.b', ':g']

    # Generate fractional resonance lines
    for n in range(1, order + 1):
        style = line_styles[n - 1]
        for i in range(0, n + 1):
            a = i
            b = n - a
            for f in range(int(-f_0 * n), int(f_0 * n)):
                if a != 0:
                    xp = [(f - b * y_min) / a + x_off, (f - b * y_max) / a + x_off]
                    plt.plot(xp, y_in, style)
                    xp = [(f + b * y_min) / a + x_off, (f + b * y_max) / a + x_off]
                    plt.plot(xp, y_in, style)
                else:
                    yp = [f / b + y_off, f / b + y_off]
                    plt.plot(x_in, yp, style)

    # Formatting plot limits and labels
    axes = plt.gca()
    axes.set_xlim(x_in)
    axes.set_ylim(y_in)
    axes.set_xlabel('Horizontal tune')
    axes.set_ylabel('Vertical tune')
    axes.set_title('Tune diagram (Blue dot = Working point)')
    plt.show()

# ==========================================
# 4. Storage Ring Lattice Specifications
# ==========================================

# Base template: MAX I 550 MeV Ring
template = LatticeClass()
template.name = "MAX I 550 MeV (template)"
template.elements = pd.DataFrame(
    {
        1:  ['Drift', 1.295,   0,        0],
        2:  ['Quad',  0.21,    0,        4.25],
        3:  ['Drift', 0.265,   0,        0],
        4:  ['Quad',  0.21,    0,       -3.02],
        5:  ['Drift', 0.27025, 0,        0],
        6:  ['Rbend', 0.9425,  0.785398, 0],
        7:  ['Drift', 0.6523,  0,        0],
        8:  ['Quad',  0.41,    0,        4.19],
        9:  ['Drift', 0.6523,  0,        3],
        10: ['Rbend', 0.9425,  0.785398, 0],
        11: ['Drift', 0.27025, 0,        0],
        12: ['Quad',  0.21,    0,       -3.02],
        13: ['Drift', 0.265,   0,        0],
        14: ['Quad',  0.21,    0,        4.25],
        15: ['Drift', 1.295,   0,        0],
    },
    index=['type', 'length', 'bending angle', 'K-value']
)

# Global compilation list
Lattices = [template]

# Option: SESAME 2.5 GeV Lattice
myNewLattice = LatticeClass()
myNewLattice.name = 'SESAME 2.5 GeV'
myNewLattice.elements = pd.DataFrame(
    {
        1:  ['Sext',  0.1,    0.0,                  7.6],
        2:  ['Drift', 0.15,   0.0,                  0.0],
        3:  ['Quad',  0.1,    0.0,                 -0.91],
        4:  ['Drift', 0.15,   0.0,                  0.0],
        5:  ['Quad',  0.3,    0.0,                  1.98],
        6:  ['Drift', 0.15,   0.0,                  0.0],
        7:  ['Sext',  0.1,    0.0,                -13.1],
        8:  ['Drift', 0.28,   0.0,                  0.0],
        9:  ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        10: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        11: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        12: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        13: ['Drift', 0.28,   0.0,                  0.0],
        14: ['Sext',  0.1,    0.0,                  7.6],
        15: ['Drift', 0.15,   0.0,                  0.0],
        16: ['Quad',  0.3,    0.0,                  1.98],
        17: ['Drift', 0.15,   0.0,                  0.0],
        18: ['Quad',  0.1,    0.0,                 -0.91],
        19: ['Drift', 0.15,   0.0,                  0.0],
        20: ['Sext',  0.1,    0.0,                -13.1],
        21: ['Drift', 2.52,   0.0,                  0.0],
        22: ['Sext',  0.1,    0.0,                  7.6],
        23: ['Drift', 0.15,   0.0,                  0.0],
        24: ['Quad',  0.1,    0.0,                 -0.91],
        25: ['Drift', 0.15,   0.0,                  0.0],
        26: ['Quad',  0.3,    0.0,                  1.98],
        27: ['Drift', 0.15,   0.0,                  0.0],
        28: ['Sext',  0.1,    0.0,                -13.1],
        29: ['Drift', 0.28,   0.0,                  0.0],
        30: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        31: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        32: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        33: ['Sbend', 0.5625, 2 * np.pi / (4 * 16), -0.335],
        34: ['Drift', 0.28,   0.0,                  0.0],
        35: ['Sext',  0.1,    0.0,                  7.6],
        36: ['Drift', 0.15,   0.0,                  0.0],
        37: ['Quad',  0.3,    0.0,                  1.98],
        38: ['Drift', 0.15,   0.0,                  0.0],
        39: ['Quad',  0.1,    0.0,                 -0.91],
        40: ['Drift', 0.15,   0.0,                  0.0],
        41: ['Sext',  0.1,    0.0,                -13.1],
        42: ['Drift', 4.3,    0.0,                  0.0]
    },
    index=['type', 'length', 'bending angle', 'K-value']
)
Lattices.append(myNewLattice)

# Option: MAX IV 3GeV Ring
miv3gev = LatticeClass()
miv3gev.name = 'MAX IV 3GeV'
miv3gev.elements = pd.DataFrame(
    {
        1:  ['Drift', 2.5,     0,                   0],
        2:  ['Drift', 0.175,   0,                   0],
        3:  ['Quad',  0.25,    0,                   3.533676],
        4:  ['Drift', 0.225,   0,                   0],
        5:  ['Quad',  0.25,    0,                  -2.23958],
        6:  ['Drift', 0.006,   0,                   0],
        7:  ['Sbend', 0.75432, 1.5 * (np.pi / 180), -0.54525],
        8:  ['Drift', 0.46268, 0,                   0],
        9:  ['Drift', 1.302,   0,                   0],
        10: ['Quad',  0.55,    0,                   2.061576],
        11: ['Drift', 0.51311, 0,                   0],
        12: ['Sbend', 1.22387, 3.0 * (np.pi / 180), -0.70480],
        13: ['Drift', 0.51311, 0,                   0],
        14: ['Quad',  0.55,    0,                   2.202441],
        15: ['Drift', 0.51311, 0,                   0],
        16: ['Sbend', 1.22378, 3.0 * (np.pi / 180), -0.70408],
        17: ['Drift', 0.51311, 0,                   0],
        18: ['Quad',  0.55,    0,                   2.202441],
        19: ['Drift', 0.51311, 0,                   0],
        20: ['Sbend', 1.22378, 3.0 * (np.pi / 180), -0.70408],
        21: ['Drift', 0.51311, 0,                   0],
        22: ['Quad',  0.55,    0,                   2.202441],
        23: ['Drift', 0.51311, 0,                   0],
        24: ['Sbend', 1.22378, 3.0 * (np.pi / 180), -0.70408],
        25: ['Drift', 0.51311, 0,                   0],
        26: ['Quad',  0.55,    0,                   2.202441],
        27: ['Drift', 0.51311, 0,                   0],
        28: ['Sbend', 1.22378, 3.0 * (np.pi / 180), -0.70408],
        29: ['Drift', 0.51311, 0,                   0],
        30: ['Quad',  0.55,    0,                   2.061576],
        31: ['Drift', 1.302,   0,                   0],
        32: ['Drift', 0.46268, 0,                   0],
        33: ['Sbend', 0.75432, 1.5 * (np.pi / 180), -0.54525],
        34: ['Drift', 0.006,   0,                   0],
        35: ['Quad',  0.25,    0,                  -2.23958],
        36: ['Drift', 0.225,   0,                   0],
        37: ['Quad',  0.25,    0,                   3.533676],
        38: ['Drift', 0.175,   0,                   0],
        39: ['Drift', 2.5,     0,                   0]
    },
    index=['type', 'length', 'bending angle', 'K-value']
)
Lattices.append(miv3gev)

# Option: Sirius 3GeV Ring
sirius = LatticeClass()
sirius.name = 'Sirius 3GeV'
sirius.elements = pd.DataFrame(
    {
        1:  ['Drift', 3.6529, 0.0,                       0.0],
        2:  ['Sext',  0.15,   0.0,                     -54.389],
        3:  ['Drift', 0.15,   0.0,                       0.0],
        4:  ['Quad',  0.2,    0.0,                      -3.56389],
        5:  ['Drift', 0.24,   0.0,                       0.0],
        6:  ['Sext',  0.15,   0.0,                      79.970],
        7:  ['Drift', 0.15,   0.0,                       0.0],
        8:  ['Quad',  0.14,   0.0,                       1.5324],
        9:  ['Drift', 0.134,  0.0,                       0.0],
        10: ['Drift', 0.13,   0.0,                       0.0],
        11: ['Sbend', 0.84,   2.9697 * np.pi / 180,      0.7693],
        12: ['Drift', 0.13,   0.0,                       0.0],
        13: ['Drift', 0.474,  0.0,                       0.0],
        14: ['Sext',  0.15,   0.0,                      79.970],
        15: ['Drift', 0.17,   0.0,                       0.0],
        16: ['Quad',  0.2,    0.0,                      -2.90469],
        17: ['Drift', 0.26,   0.0,                       0.0],
        18: ['Sext',  0.15,   0.0,                    -228.397],
        19: ['Drift', 0.23,   0.0,                       0.0],
        20: ['Quad',  0.2,    0.0,                      -4.265782],
        21: ['Drift', 0.17,   0.0,                       0.0],
        22: ['Sext',  0.15,   0.0,                     126.648],
        23: ['Drift', 0.216,  0.0,                       0.0],
        24: ['Drift', 0.1145, 0.0,                       0.0],
        25: ['Sbend', 1.25,   4.4192 * np.pi / 180,      0.76842],
        26: ['Drift', 0.1145, 0.0,                       0.0],
        27: ['Drift', 0.336,  0.0,                       0.0],
        28: ['Sext',  0.15,   0.0,                     182.964],
        29: ['Drift', 0.17,   0.0,                       0.0],
        30: ['Quad',  0.2,    0.0,                      -3.282953],
        31: ['Drift', 0.23,   0.0,                       0.0],
        32: ['Sext',  0.15,   0.0,                    -213.397],
        33: ['Drift', 0.26,   0.0,                       0.0],
        34: ['Quad',  0.2,    0.0,                      -3.8691],
        35: ['Drift', 0.827,  0.0,                       0.0],
        36: ['Drift', 0.065,  0.0,                       0.0],
        37: ['Sbend', 0.436,  1.5414 * np.pi / 180,      0.71651],
        38: ['Sbend', 0.04,   0.1414 * np.pi / 180,      0.0],
        39: ['Sbend', 0.436,  1.5414 * np.pi / 180,      0.71651],
        40: ['Drift', 0.065,  0.0,                       0.0],
        41: ['Drift', 0.336,  0.0,                       0.0],
        42: ['Sext',  0.15,   0.0,                     182.964],
        43: ['Drift', 0.17,   0.0,                       0.0],
        44: ['Quad',  0.2,    0.0,                      -3.282953],
        45: ['Drift', 0.23,   0.0,                       0.0],
        46: ['Sext',  0.15,   0.0,                    -213.397],
        47: ['Drift', 0.26,   0.0,                       0.0],
        48: ['Quad',  0.2,    0.0,                      -3.8691],
        49: ['Drift', 0.827,  0.0,                       0.0],
        50: ['Drift', 0.1145, 0.0,                       0.0],
        51: ['Sbend', 1.25,   4.4192 * np.pi / 180,      0.76842],
        52: ['Drift', 0.1145, 0.0,                       0.0],
        53: ['Drift', 0.474,  0.0,                       0.0],
        54: ['Sext',  0.15,   0.0,                      79.970],
        55: ['Drift', 0.17,   0.0,                       0.0],
        56: ['Quad',  0.2,    0.0,                      -2.90469],
        57: ['Drift', 0.26,   0.0,                       0.0],
        58: ['Sext',  0.15,   0.0,                    -228.397],
        59: ['Drift', 0.23,   0.0,                       0.0],
        60: ['Quad',  0.2,    0.0,                      -4.265782],
        61: ['Drift', 0.17,   0.0,                       0.0],
        62: ['Sext',  0.15,   0.0,                     126.648],
        63: ['Drift', 0.216,  0.0,                       0.0],
        64: ['Drift', 0.13,   0.0,                       0.0],
        65: ['Sbend', 0.84,   2.9697 * np.pi / 180,      0.7693],
        66: ['Drift', 0.13,   0.0,                       0.0],
        67: ['Drift', 0.134,  0.0,                       0.0],
        68: ['Quad',  0.14,   0.0,                       2.041701],
        69: ['Drift', 0.15,   0.0,                       0.0],
        70: ['Sext',  0.15,   0.0,                     -73.665],
        71: ['Drift', 0.24,   0.0,                       0.0],
        72: ['Quad',  0.3,    0.0,                      -4.098337],
        73: ['Drift', 0.15,   0.0,                       0.0],
        74: ['Sext',  0.15,   0.0,                     -73.665],
        75: ['Drift', 0.185,  0.0,                       0.0],
        76: ['Quad',  0.14,   0.0,                       3.285511],
        77: ['Drift', 0.23,   0.0,                       0.0]
    }
)
Lattices.append(sirius)


In [ ]:
#@title #Lattice initialisation
import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. UI Configurations
# ==========================================

options = [lat.name for lat in Lattices]

latticeSelect = widgets.RadioButtons(
    options=options,
    description='Select lattice:',
    disabled=False
)

energy_widget = widgets.FloatText(
    value=2.5,
    description='Energy =',
    disabled=False,
    layout=widgets.Layout(width='180px')
)

energy_unit = widgets.Label(
    value='GeV',
    layout=widgets.Layout(margin='0 0 0 5px')
)

energy_layout = widgets.HBox([energy_widget, energy_unit])

period = widgets.IntText(
    value=8,
    description='Periodicity =',
    disabled=False,
    layout=widgets.Layout(width='180px')
)

period_unit = widgets.Label(
    value=' ',
    layout=widgets.Layout(margin='0 0 0 5px')
)

period_layout = widgets.HBox([period, period_unit])

buttonMakeLattice = widgets.Button(description="Make the Lattice")
outputMakeLattice = widgets.Output()

# Initialize global placeholders
lattice = 0
refpts = 0

# ==========================================
# 2. Lattice Building
# ==========================================

def on_buttonMakeLattice_clicked(b):
    """Generates the AT tracking lattice based on user widget selections."""
    with outputMakeLattice:
        clear_output()

        global lattice, lattice_t, num_of_elem, THERING, length, refpts, s, optics

        selected_idx = options.index(latticeSelect.value)
        lattice_in = Lattices[selected_idx].elements
        lattice_t = lattice_in.transpose()
        elements = lattice_t.values
        num_of_elem = len(lattice_in.columns)

        # Build individual AT element objects
        lattice_elements = []
        for i in range(num_of_elem):
            elem_type = elements[i][0]
            length_val = elements[i][1]
            angle = elements[i][2]
            k_val = elements[i][3]

            if elem_type == 'Drift':
                elem = at.elements.Drift('drift', length_val)
            elif elem_type == 'Quad':
                elem = at.elements.Quadrupole('quad', length_val, k_val)
                elem.PassMethod = 'QuadLinearPass'
            elif elem_type == 'Rbend':
                elem = at.elements.Bend('dip', length_val, angle)
                elem.K = k_val
                elem.EntranceAngle = angle / 2
                elem.ExitAngle = angle / 2
                elem.PassMethod = 'BendLinearPass'
            elif elem_type == 'Sbend':
                elem = at.elements.Bend('dip', length_val, angle)
                elem.K = k_val
                elem.EntranceAngle = 0
                elem.ExitAngle = 0
                elem.PassMethod = 'BendLinearPass'
            elif elem_type == 'Sext':
                elem = at.elements.Sextupole('sext', length_val, k_val)

            lattice_elements.append(elem)

        # Expand lattice by defined periodic cells
        lattice = lattice_elements * period.value
        num_of_elem = len(lattice)
        print(f'Lattice ready with {num_of_elem} elements.')

        # Initialize AT Lattice context and calculate optics parameters
        THERING = at.lattice.Lattice(lattice, energy=energy_widget.value * 1e9, periodicity=period.value)
        length = np.size(THERING)
        refpts = np.r_[0:length + 1]
        s = at.lattice.get_s_pos(THERING)

        try:
            optics = at.physics.linopt(THERING, refpts=refpts, get_chrom=True)
            data.beta = optics[3]['beta']
            data.disp_fu = optics[3]['dispersion']
            data.alpha = optics[3]['alpha']
            data.mu = optics[3]['mu']
            data.gamma = optics[3]['gamma']
            data.s = s
        except ValueError:
            print("Oops! That was not a valid lattice. Try again...")

buttonMakeLattice.on_click(on_buttonMakeLattice_clicked)

# ==========================================
# 3. Matrix & Information Display Panels
# ==========================================

buttonCompactMatrix = widgets.Button(description="Compact")
buttonCompleteMatrix = widgets.Button(description="Complete")
buttonClearOutput = widgets.Button(description="Clear output")

outputCompleteMatrix = widgets.Output(
    layout=widgets.Layout(
        padding='15px',
        min_height='250px',
        border='1px solid #eee',
        border_radius='4px',
        background_color='#fafafa'
    )
)

def on_complete_lattice_button_clicked(b):
    with outputWholeMatrix:
        clear_output()
        try:
            a = at.find_m44(THERING, 0)
            print(a[0])
        except Exception as e:
            print(f'Error: {e}')

def on_buttonCompleteMatrix_clicked(b):
    with outputCompleteMatrix:
        try:
            for i in range(num_of_elem):
                print(lattice[i])
        except NameError:
            print('No lattice defined')

def on_buttonCompactMatrix_clicked(b):
    with outputCompleteMatrix:
        try:
            print(lattice_t)
        except NameError:
            print('No lattice defined')

def on_buttonClearOutput_clicked(b):
    with outputCompleteMatrix:
        clear_output()

buttonCompleteMatrix.on_click(on_buttonCompleteMatrix_clicked)
buttonCompactMatrix.on_click(on_buttonCompactMatrix_clicked)
buttonClearOutput.on_click(on_buttonClearOutput_clicked)

buttonElementMatrix = widgets.Button(description="Print matrix of")
outputElementMatrix = widgets.Output()

def on_element_lattice_button_clicked(b):
    with outputElementMatrix:
        clear_output()
        element = w1.value
        try:
            m66 = at.find_elem_m66(lattice[element - 1])
            print(f'Transfer matrix of element {element} = ')
            print(m66[0:4, 0:4])
        except TypeError:
            print('Error computing transfer matrix.')
        except IndexError:
            print(f'There are only {length} elements')

buttonElementMatrix.on_click(on_element_lattice_button_clicked)

buttonWholeMatrix = widgets.Button(description="Print the full lattice matrix")
outputWholeMatrix = widgets.Output()
buttonWholeMatrix.on_click(on_complete_lattice_button_clicked)

w1 = widgets.BoundedIntText(value=1, min=1, step=1, description='Element:', disabled=False)
single_element = widgets.HBox([buttonElementMatrix, w1])

# ==========================================
# 4. Geometry Plotting
# ==========================================

geometry_canvas = widgets.Output(layout=widgets.Layout(margin='5px 10px 15px 0px', min_height='100px'))
geometry_canvas2 = widgets.Output(layout=widgets.Layout(margin='5px 10px 15px 0px', min_height='100px'))
button_geom = widgets.Button(description="Plot Geometry", button_style='info', icon='image')

def on_plot_geometry_clicked(b):
    """Plots both internal Accelerator Toolbox and custom single-period geometry curves."""
    if 'THERING' not in globals() or num_of_elem == 0:
        with geometry_canvas:
            clear_output(wait=True)
            print("No lattice defined yet. Please click 'Make the Lattice' first.")
        with geometry_canvas2:
            clear_output(wait=True)
        return

    with geometry_canvas:
        clear_output(wait=True)
        try:
            fig1 = plt.figure(figsize=(8, 4))
            THERING.plot_geometry()
            plt.show()
        except Exception as e:
            pass

    with geometry_canvas2:
        clear_output(wait=True)
        try:
            geometry = np.zeros((4, 2 * num_of_elem + 2))

            for i in range(num_of_elem):
                geometry[0, 2 * i] = geometry[0, 2 * i - 1]
                geometry[0, 2 * i + 1] = lattice[i].Length + geometry[0, 2 * i]

                if lattice[i].FamName in ['quad', 'dip']:
                    geometry[1, 2 * i] = 1.5 if lattice[i].FamName == 'quad' else 2.0
                    geometry[1, 2 * i + 1] = geometry[1, 2 * i]
                    geometry[3, 2 * i] = lattice[i].K
                    geometry[3, 2 * i + 1] = lattice[i].K

            geometry[0, 2 * i + 2] = geometry[0, 2 * i + 1]

            kMax = max(geometry[3, :])
            if kMax != 0:
                geometry[3, :] /= kMax

            fig2 = plt.figure(figsize=(8, 4))
            slice_idx = (len(geometry[0]) // period.value) + 5
            plt.plot(geometry[0][:slice_idx], -geometry[1][:slice_idx], 'b')

            axes = plt.gca()
            axes.set_xlabel('s (m)')
            axes.set_title('Geometry of single period')
            plt.show()
        except Exception as e:
            print(f'Error rendering custom geometry: {e}')

button_geom.on_click(on_plot_geometry_clicked)

# ==========================================
# 5. Display Layouts
# ==========================================

matrices_tab = widgets.VBox(
    [single_element, outputElementMatrix, buttonWholeMatrix, outputWholeMatrix],
    layout=widgets.Layout(margin='5px 0px 15px 0px', justify_content='flex-start')
)

tab2 = widgets.HBox(
    [buttonCompactMatrix, buttonCompleteMatrix, buttonClearOutput],
    layout=widgets.Layout(margin='5px 0px 15px 0px', justify_content='flex-start')
)

geometry_tab_container = widgets.VBox([button_geom, geometry_canvas, geometry_canvas2])
tab2_container = widgets.VBox([tab2, outputCompleteMatrix])

the_tabs = widgets.Tab()
the_tabs.children = [tab2_container, geometry_tab_container, matrices_tab]
the_tabs.set_title(0, 'List of elements')
the_tabs.set_title(1, 'Lattice geometry')
the_tabs.set_title(2, 'Matrices')

choose_lattice_panel = widgets.VBox(
    [latticeSelect, period_layout, energy_layout, buttonMakeLattice, outputMakeLattice],
    layout=widgets.Layout(
        width='300px', min_width='220px', max_width='500px',
        overflow='hidden', resize='horizontal',
        background_color='#f7f7f7', border='1px solid #ddd',
        border_radius='4px', padding='15px', margin='0px 15px 0px 0px'
    )
)

whole_interface = widgets.HBox(
    [choose_lattice_panel, the_tabs],
    layout=widgets.Layout(width='100%', align_items='stretch')
)

# Render Dashboard
display(whole_interface)

In [ ]:
#@title #Optics calculation
import ipywidgets as widgets
from IPython.display import clear_output
import numpy as np

# ==========================================
# 1. HTML/CSS Layout Configuration
# ==========================================

TABLE_STYLE = """
<style>
    .optics-table {
        font-family: Arial, sans-serif;
        border-collapse: collapse;
        width: 100%;
        max-width: 500px;
        margin: 10px 0;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        border-radius: 4px;
        overflow: hidden;
    }
    .optics-table td, .optics-table th {
        border: 1px solid #e0e0e0;
        padding: 10px 12px;
        text-align: left;
    }
    .optics-table tr:hover { background-color: orange; }
    .optics-table th {
        padding-top: 12px;
        padding-bottom: 12px;
        background-color: #2196F3;
        color: white;
        font-weight: bold;
    }
    .error-msg {
        color: #d32f2f;
        background-color: #ffeef0;
        padding: 10px;
        border-left: 4px solid #d32f2f;
        font-weight: bold;
        border-radius: 4px;
    }
</style>
"""

# Initial placeholder for the opticspanel
optics_table_widget = widgets.HTML(
    value="<p style='color: #777; font-style: italic;'>Click 'Calculate optics' to view results.</p>"
)

# ==========================================
# 2. Optics Computation
# ==========================================

def on_button_clicked_optic(b):
    """Computes basic ring optics and yields a structured HTML presentation layout."""
    with output_optics:
        clear_output(wait=True)
        try:
            _, tunes, chroms, twiss = THERING.linopt(0.0, 0, get_chrom=True, coupled=False)
            tunes_full = optics[3]['mu'][length] / 2 / np.pi

            tunes_str = f"[{tunes_full[0]:.4f}, {tunes_full[1]:.4f}]"
            chrom_str = f"[{chroms[0]:.4f}, {chroms[1]:.4f}]"
            circum_str = f"{data.s[-1]:.4f} m"

            table_html = f"""
            {TABLE_STYLE}
            <table class="optics-table">
                <thead>
                    <tr>
                        <th>Parameter</th>
                        <th>Value</th>
                    </tr>
                </thead>
                <tbody>
                    <tr>
                        <td><b>Tunes (v_x, v_y)</b></td>
                        <td>{tunes_str}</td>
                    </tr>
                    <tr>
                        <td><b>Circumference</b></td>
                        <td>{circum_str}</td>
                    </tr>
                    <tr>
                        <td><b>Chromaticity (xi_x, xi_y)</b></td>
                        <td>{chrom_str}</td>
                    </tr>
                </tbody>
            </table>
            """
            optics_table_widget.value = table_html

        except NameError:
            optics_table_widget.value = f"{TABLE_STYLE}<div class='error-msg'> Error: No lattice defined</div>"
        except ValueError:
            optics_table_widget.value = f"{TABLE_STYLE}<div class='error-msg'> Error: Not a valid (segment of a) ring</div>"

# ==========================================
# 3. Widget Initialization
# ==========================================

button_optic = widgets.Button(
    description="Calculate optics",
    button_style='success',
    icon='calculator',
    layout=widgets.Layout(margin='10px 0px')
)

output_optics = widgets.Output()
button_optic.on_click(on_button_clicked_optic)

optics_tab_container = widgets.VBox([
    button_optic,
    optics_table_widget,
    output_optics
])

# Render Dashboard Component
display(optics_tab_container)

In [ ]:
#@title #Optic functions
import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt

# ==========================================
# 1. UI Setup
# ==========================================

btn_layout = widgets.Layout(width='100%')

buttonDisp = widgets.Button(description="Plot dispersion", button_style='primary', layout=btn_layout)
outputDispFunction = widgets.Output()

buttonBeta = widgets.Button(description="Plot beta functions", button_style='primary', layout=btn_layout)
outputBetaFunction = widgets.Output()

buttonAlpha = widgets.Button(description="Plot alpha functions", button_style='primary', layout=btn_layout)
outputAlphaFunction = widgets.Output()

buttonGamma = widgets.Button(description="Plot gamma function", button_style='primary', layout=btn_layout)
outputGammaFunction = widgets.Output()

buttonMu = widgets.Button(description="Plot mu functions", button_style='primary', layout=btn_layout)
outputMuFunction = widgets.Output()

calculate_all_button = widgets.Button(description="Plot all", button_style='success', icon='chart-bar', layout=btn_layout)
clear_all_button = widgets.Button(description="Clean all", button_style='danger', icon='trash', layout=btn_layout)

# Global layout config
container_layout = widgets.Layout(align_items='center', justify_content='center', padding='10px')

# ==========================================
# 2. Plot Functions
# ==========================================

def on_disp_button_clicked(b):
    with outputDispFunction:
        clear_output(wait=True)
        if data.disp_fu is None:
            print('No lattice defined or optics not calculated yet!')
            return
        plt.figure(figsize=(7, 3.5))
        plt.plot(data.s, data.disp_fu[:, 0], 'r')
        axes = plt.gca()
        axes.set_xlabel('s (m)')
        axes.set_ylabel('dispersion (m)')
        axes.set_title('Dispersion function')
        plt.show()

def on_beta_button_clicked(b):
    with outputBetaFunction:
        clear_output(wait=True)
        if data.beta is None:
            print('No lattice defined or optics not calculated yet!')
            return
        plt.figure(figsize=(7, 3.5))
        l1, = plt.plot(data.s, data.beta[:, 0], color='green')
        l2, = plt.plot(data.s, data.beta[:, 1])
        axes = plt.gca()
        axes.set_xlabel('s (m)')
        axes.set_ylabel('beta (m)')
        axes.set_title('Beta functions')
        plt.legend([l1, l2], ['Beta x', 'Beta y'])
        plt.show()

def on_alpha_button_clicked(b):
    with outputAlphaFunction:
        clear_output(wait=True)
        if data.alpha is None:
            print('No lattice defined or optics not calculated yet!')
            return
        plt.figure(figsize=(7, 3.5))
        l1, = plt.plot(data.s, data.alpha[:, 0], color='green')
        l2, = plt.plot(data.s, data.alpha[:, 1])
        axes = plt.gca()
        axes.set_xlabel('s (m)')
        axes.set_ylabel('alpha (m)')
        axes.set_title('Alpha functions')
        plt.legend([l1, l2], ['Alpha x', 'Alpha y'])
        plt.show()

def on_gamma_button_clicked(b):
    with outputGammaFunction:
        clear_output(wait=True)
        if data.gamma is None:
            print('No lattice defined or optics not calculated yet!')
            return
        plt.figure(figsize=(7, 3.5))
        l1, = plt.plot(data.s, data.gamma[:], color='green')
        axes = plt.gca()
        axes.set_xlabel('s (m)')
        axes.set_ylabel('gamma (m)')
        axes.set_title('Gamma functions')
        plt.legend([l1], ['Gamma'])
        plt.show()

def on_mu_button_clicked(b):
    with outputMuFunction:
        clear_output(wait=True)
        if data.mu is None:
            print('No lattice defined or optics not calculated yet!')
            return
        plt.figure(figsize=(7, 3.5))
        l1, = plt.plot(data.s, data.mu[:, 0], color='green')
        l2, = plt.plot(data.s, data.mu[:, 1])
        axes = plt.gca()
        axes.set_xlabel('s (m)')
        axes.set_ylabel('mu (m)')
        axes.set_title('Mu functions')
        plt.legend([l1, l2], ['Mu x', 'Mu y'])
        plt.show()

def on_plot_all(b):
    """Triggers individual plotting callbacks sequentially."""
    on_disp_button_clicked(None)
    on_beta_button_clicked(None)
    on_alpha_button_clicked(None)
    on_gamma_button_clicked(None)
    on_mu_button_clicked(None)

def on_clear_all(b):
    """Clears all display canvas containers."""
    with outputDispFunction: clear_output()
    with outputBetaFunction: clear_output()
    with outputAlphaFunction: clear_output()
    with outputGammaFunction: clear_output()
    with outputMuFunction: clear_output()

# ==========================================
# 3. Click Event
# ==========================================

buttonDisp.on_click(on_disp_button_clicked)
buttonBeta.on_click(on_beta_button_clicked)
buttonAlpha.on_click(on_alpha_button_clicked)
buttonGamma.on_click(on_gamma_button_clicked)
buttonMu.on_click(on_mu_button_clicked)
calculate_all_button.on_click(on_plot_all)
clear_all_button.on_click(on_clear_all)

# ==========================================
# 4. Layout Grouping
# ==========================================

container_1 = widgets.VBox([buttonDisp, outputDispFunction], layout=container_layout)
container_2 = widgets.VBox([buttonBeta, outputBetaFunction], layout=container_layout)
container_3 = widgets.VBox([buttonAlpha, outputAlphaFunction], layout=container_layout)
container_4 = widgets.VBox([buttonGamma, outputGammaFunction], layout=container_layout)
container_5 = widgets.VBox([buttonMu, outputMuFunction], layout=container_layout)

options_panel = widgets.VBox(
    [calculate_all_button, clear_all_button],
    layout=widgets.Layout(
        width='150px',
        align_items='center',
        justify_content='flex-start',
        padding='10px 5px',
        grid_gap='5px'
    )
)

the_tabs2 = widgets.Tab()
the_tabs2.children = [container_1, container_2, container_3, container_4, container_5]
the_tabs2.set_title(0, 'Dispersion')
the_tabs2.set_title(1, 'Beta function')
the_tabs2.set_title(2, 'Alpha function')
the_tabs2.set_title(3, 'Gamma function')
the_tabs2.set_title(4, 'Mu function')

whole_interface = widgets.HBox(
    [options_panel, the_tabs2],
    layout=widgets.Layout(width='100%', align_items='stretch')
)

# Render plotting control center
display(whole_interface)

In [ ]:
#@title #Tune diagram

import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. Slider & Widget Initialization
# ==========================================

x_range_plot = widgets.IntRangeSlider(
    value=[0, 2], min=0, max=40, step=1,
    description='X Range:',
    continuous_update=True,
    layout=widgets.Layout(width='100%')
)

y_range_plot = widgets.IntRangeSlider(
    value=[0, 2], min=0, max=40, step=1,
    description='Y Range:',
    continuous_update=True,
    layout=widgets.Layout(width='100%')
)

order_slider_plot = widgets.IntSlider(
    value=2, min=1, max=4, step=1,
    description='Res. Order:',
    continuous_update=True,
    layout=widgets.Layout(width='100%')
)

button_refresh_plot = widgets.Button(
    description="Force Refresh",
    button_style='info',
    icon='refresh',
    layout=widgets.Layout(width='100%')
)

button_clear_plot = widgets.Button(
    description="Clear plot",
    button_style='warning',
    icon='trash',
    layout=widgets.Layout(width='100%')
)

output_area_plot = widgets.Output(
    layout=widgets.Layout(
        padding='10px',
        min_width='450px',
        align_items='center',
        justify_content='center'
    )
)

# ==========================================
# 2. Resonance Lines Plotting
# ==========================================

def execute_tune_space_plot():
    """Generates the fractional tune working point map and resonance lines."""
    try:
        x_limits_p = list(x_range_plot.value)
        y_limits_p = list(y_range_plot.value)

        # Compute whole ring tune configurations from phase advances
        tunes_full_p = optics[3]['mu'][length] / 2 / np.pi

        plt.figure(figsize=(6, 6))
        tunespace(tunes_full_p, x_limits_p, y_limits_p, order_slider_plot.value)
        plt.show()

    except NameError:
        print('No lattice defined or optics not calculated yet. Please check step 1.')

def handle_setting_change_plot(change):
    """Refreshes the canvas dynamically on slider adjustments."""
    with output_area_plot:
        clear_output(wait=True)
        execute_tune_space_plot()

def handle_refresh_click_plot(b):
    with output_area_plot:
        clear_output(wait=True)
        execute_tune_space_plot()

def handle_clear_click_plot(b):
    with output_area_plot:
        clear_output()

# ==========================================
# 3. Signals & Observers Linking
# ==========================================

x_range_plot.observe(handle_setting_change_plot, names='value')
y_range_plot.observe(handle_setting_change_plot, names='value')
order_slider_plot.observe(handle_setting_change_plot, names='value')

button_refresh_plot.on_click(handle_refresh_click_plot)
button_clear_plot.on_click(handle_clear_click_plot)

# ==========================================
# 4. Interface Construction
# ==========================================

settings_panel_plot = widgets.VBox(
    [
        widgets.HTML("<b>Tune Space Limits</b>"),
        x_range_plot,
        y_range_plot,
        order_slider_plot,
        widgets.Box(layout=widgets.Layout(height='15px')),
        button_refresh_plot,
        button_clear_plot
    ],
    layout=widgets.Layout(
        width='320px',
        min_width='280px',
        padding='15px',
        border='1px solid #ddd',
        border_radius='4px',
        background_color='#fcfcfc',
        grid_gap='8px'
    )
)

plot_panel_plot = widgets.VBox(
    [output_area_plot],
    layout=widgets.Layout(
        flex='1 1 auto',
        align_items='center',
        justify_content='center'
    )
)

tune_space_main_interface = widgets.HBox(
    [settings_panel_plot, plot_panel_plot],
    layout=widgets.Layout(width='100%', align_items='stretch', padding='10px')
)

# Render Tune Workspace Diagram Applet
display(tune_space_main_interface)

In [ ]:
#@title ###Single Particle Tracking

import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. Sliders Initialisation
# ==========================================

slider_x = widgets.FloatSlider(value=0.5, min=0, max=5.0, step=0.1, description="x (mm):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_xp = widgets.FloatSlider(value=0.2, min=0, max=2.0, step=0.05, description="x' (mRad):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_y = widgets.FloatSlider(value=0.5, min=0, max=5.0, step=0.1, description="y (mm):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_yp = widgets.FloatSlider(value=0.2, min=0, max=2.0, step=0.05, description="y' (mRad):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_dp = widgets.FloatSlider(value=0.0, min=0, max=5.0, step=0.1, description="dp/p (0/00):", continuous_update=True, layout=widgets.Layout(width='100%'))

button_track1 = widgets.Button(
    description="Force Track Single",
    button_style='info',
    icon='circle',
    layout=widgets.Layout(width='100%')
)

output_track1 = widgets.Output(layout=widgets.Layout(align_items='center', justify_content='center'))

# ==========================================
# 2. Tracking Logic
# ==========================================

def run_single_track():
    """Tracks a single particle through the element structure and updates the canvas."""
    with output_track1:
        clear_output(wait=True)
        try:
            # Construct 6D initial coordinate vector (scaling to SI units: meters / radians)
            X0 = np.zeros((6, 1))
            X0[0] = slider_x.value * 1e-3
            X0[1] = slider_xp.value * 1e-3
            X0[2] = slider_y.value * 1e-3
            X0[3] = slider_yp.value * 1e-3
            X0[4] = slider_dp.value * 1e-3

            X_out = at.lattice_pass(lattice, X0, nturns=1, refpts=refpts)

            plt.figure(figsize=(10, 3.5))
            plt.plot(s, X_out[0, 0, :, 0], 'b', label='x offset')
            plt.plot(s, X_out[2, 0, :, 0], 'r', label='y offset')
            plt.xlabel('s (m)')
            plt.ylabel('offset (m)')
            plt.title('Single Particle Trajectory')
            plt.legend()
            plt.grid(True, linestyle=':')
            plt.show()

        except (TypeError, NameError) as e:
            print(f'No valid lattice defined or optics calculated yet.{e}')

def on_single_change(change):
    """Callback linked to dynamic slider updates."""
    run_single_track()

# Bind reactive signals specifically to value changes
for slider in [slider_x, slider_xp, slider_y, slider_yp, slider_dp]:
    slider.observe(on_single_change, names='value')

button_track1.on_click(lambda b: run_single_track())

# ==========================================
# 3. Interface Packing
# ==========================================

left_column = widgets.VBox(
    [
        widgets.HTML("<h3>Single Particle Tracking</h3>"),
        slider_x, slider_xp, slider_y, slider_yp, slider_dp,
        widgets.Box(layout=widgets.Layout(height='10px', width='300px')),
        button_track1,
        widgets.Box(layout=widgets.Layout(height='10px', width='300px')),
        output_track1
    ],
    layout=widgets.Layout(
        flex='1 1 0%',
        padding='15px',
        border='1px solid #ddd',
        border_radius='6px',
        background_color='#f9fbfd',
        grid_gap='5px',
        width='300px'
    )
)

tracking_tab_interface = widgets.HBox(
    [left_column],
    layout=widgets.Layout(
        width='100%',
        align_items='stretch',
        grid_gap='15px',
        padding='10px'
    )
)

# Render Single-Particle Tracking Component
display(tracking_tab_interface)

In [ ]:
#@title ###Multiple Particle Tracking

import ipywidgets as widgets
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
from time import sleep

# ==========================================
# 1. State Variables & Interface Output Canvas Blocks
# ==========================================

beam_tracking_data = None

center_layout = widgets.Layout(
    align_items='center',
    justify_content='center',
    padding='10px',
    width='100%'
)

out_real_space = widgets.Output()
out_phase_space = widgets.Output()
out_envelope = widgets.Output()
out_emittance = widgets.Output()
out_status = widgets.Output()
out_emittance_calc = widgets.Output(layout=widgets.Layout(align_items='center', justify_content='center'))

# ==========================================
# 2. Control Slider Adjusters Initialization
# ==========================================

slider_x_rms = widgets.FloatSlider(value=0.5, min=0, max=5.0, step=0.1, description="x rms (mm):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_xp_rms = widgets.FloatSlider(value=0.2, min=0, max=2.0, step=0.05, description="x' rms (mRad):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_y_rms = widgets.FloatSlider(value=0.5, min=0, max=5.0, step=0.1, description="y rms (mm):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_yp_rms = widgets.FloatSlider(value=0.2, min=0, max=2.0, step=0.05, description="y' rms (mRad):", continuous_update=True, layout=widgets.Layout(width='100%'))
slider_dp_rms = widgets.FloatSlider(value=0.1, min=0, max=5.0, step=0.1, description="dp/p rms:", continuous_update=True, layout=widgets.Layout(width='100%'))

input_emit_x = widgets.BoundedFloatText(value=0.0, min=0, description="X emit (um):", layout=widgets.Layout(width='100%'))
input_emit_y = widgets.BoundedFloatText(value=0.0, min=0, description="Y emit (um):", layout=widgets.Layout(width='100%'))
btn_calculate_from_emit = widgets.Button(description="Calculate from Emit", button_style='warning', icon='calculator', layout=widgets.Layout(width='100%'))

def calculate_rms_from_emittance(b):
    """Calculates equivalent RMS sizing coordinates using linear Twiss beta parameters."""
    with out_emittance_calc:
        clear_output()
        try:
            slider_x_rms.value = np.sqrt(input_emit_x.value * 1e-6 * data.beta[0][0]) * 1e3
            slider_xp_rms.value = np.sqrt(input_emit_x.value * 1e-6 / data.beta[0][0]) * 1e3
            slider_y_rms.value = np.sqrt(input_emit_y.value * 1e-6 * data.beta[0][1]) * 1e3
            slider_yp_rms.value = np.sqrt(input_emit_y.value * 1e-6 / data.beta[0][1]) * 1e3
        except Exception as e:
            print(f'Error! {e}')

btn_calculate_from_emit.on_click(calculate_rms_from_emittance)

# ==========================================
# 3. Canvas Visualization Engines
# ==========================================

def plot_real_space(element_index):
    sleep(20)
    with out_real_space:
        clear_output(wait=True)
        if beam_tracking_data is None:
            return

        plt.figure(figsize=(5, 4))
        plt.plot(beam_tracking_data[0, :, element_index, 0], beam_tracking_data[2, :, element_index, 0], '.', markersize=2)
        axes = plt.gca()
        axes.set_xlabel('x (m)')
        axes.set_ylabel('y (m)')
        axes.set_title('Real space')

        xmin, xmax = np.min(beam_tracking_data[0,:,:,0]), np.max(beam_tracking_data[0,:,:,0])
        if xmin == xmax: xmin, xmax = xmin-0.05, xmax+0.05
        xmax = np.max([np.abs(xmin), np.abs(xmax)])
        axes.set_xlim([-xmax, xmax])

        ymin, ymax = np.min(beam_tracking_data[2,:,:,0]), np.max(beam_tracking_data[2,:,:,0])
        if ymin == ymax: ymin, ymax = ymin-0.05, ymax+0.05
        ymax = np.max([np.abs(ymin), np.abs(ymax)])
        axes.set_ylim([-ymax, ymax])
        plt.grid(True, linestyle=':')
        plt.show()

def plot_phase_space(element_index):
    with out_phase_space:
        clear_output(wait=True)
        if beam_tracking_data is None:
            print("beam_tracking_data is none")
            return

        plt.figure(figsize=(10, 3.8))

        # Horizontal Axis Block
        plt.subplot(1, 2, 1)
        plt.plot(beam_tracking_data[0, :, element_index, 0], beam_tracking_data[1, :, element_index, 0], '.', markersize=2)
        axes = plt.gca()
        axes.set_xlabel('x (m)')
        axes.set_ylabel('xp (rad)')
        axes.set_title('Horizontal phase space')
        xmin, xmax = np.min(beam_tracking_data[0,:,:,0]), np.max(beam_tracking_data[0,:,:,0])
        if xmin == xmax: xmin, xmax = xmin-0.05, xmax+0.05
        xmax = np.max([np.abs(xmin), np.abs(xmax)])
        ymin, ymax = np.min(beam_tracking_data[1,:,:,0]), np.max(beam_tracking_data[1,:,:,0])
        if ymin == ymax: ymin, ymax = ymin-0.05, ymax+0.05
        ymax = np.max([np.abs(ymin), np.abs(ymax)])
        axes.set_xlim([-xmax, xmax])
        axes.set_ylim([-ymax, ymax])
        plt.grid(True, linestyle=':')

        # Vertical Axis Block
        plt.subplot(1, 2, 2)
        plt.plot(beam_tracking_data[2, :, element_index, 0], beam_tracking_data[3, :, element_index, 0], '.', markersize=2, color='orange')
        axes = plt.gca()
        axes.set_xlabel('y (m)')
        axes.set_ylabel('yp (rad)')
        axes.set_title('Vertical phase space')
        xmin, xmax = np.min(beam_tracking_data[2,:,:,0]), np.max(beam_tracking_data[2,:,:,0])
        if xmin == xmax: xmin, xmax = xmin-0.05, xmax+0.05
        xmax = np.max([np.abs(xmin), np.abs(xmax)])
        ymin, ymax = np.min(beam_tracking_data[3,:,:,0]), np.max(beam_tracking_data[3,:,:,0])
        if ymin == ymax: ymin, ymax = ymin-0.05, ymax+0.05
        ymax = np.max([np.abs(ymin), np.abs(ymax)])
        axes.set_xlim([-xmax, xmax])
        axes.set_ylim([-ymax, ymax])
        plt.grid(True, linestyle=':')

        plt.tight_layout()
        plt.show()

def plot_beam_envelope():
    with out_envelope:
        clear_output(wait=True)
        if beam_tracking_data is None:
            return

        s_plot = s[:beam_tracking_data.shape[2]]
        x_max = np.max(beam_tracking_data[0, :, :, 0], axis=0)
        x_min = np.min(beam_tracking_data[0, :, :, 0], axis=0)
        y_max = np.max(beam_tracking_data[2, :, :, 0], axis=0)
        y_min = np.min(beam_tracking_data[2, :, :, 0], axis=0)

        plt.figure(figsize=(8, 4.5))
        plt.plot(s_plot, x_max, label='x max', color='blue')
        plt.plot(s_plot, x_min, color='blue', alpha=0.4)
        plt.plot(s_plot, y_max, label='y max', color='orange')
        plt.plot(s_plot, y_min, color='orange', alpha=0.4)

        plt.fill_between(s_plot, x_min, x_max, color='blue', alpha=0.15)
        plt.fill_between(s_plot, y_min, y_max, color='orange', alpha=0.15)

        plt.xlabel('s (m)')
        plt.ylabel('Beam size (m)')
        plt.title('Beam envelope along lattice')
        plt.legend()
        plt.grid(True, linestyle=':')
        plt.show()

def calculate_statistical_emittance(X):
    x, xp, y, yp = X[0,:], X[1,:], X[2,:], X[3,:]
    eps_x = np.sqrt(np.mean(x**2) * np.mean(xp**2) - np.mean(x * xp)**2)
    eps_y = np.sqrt(np.mean(y**2) * np.mean(yp**2) - np.mean(y * yp)**2)
    return eps_x, eps_y

def plot_emittance_evolution():
    with out_emittance:
        clear_output(wait=True)
        if beam_tracking_data is None:
            return

        eps_x_list, eps_y_list = [], []
        for r in range(beam_tracking_data.shape[2]):
            X = beam_tracking_data[:, :, r, 0]
            ex, ey = calculate_statistical_emittance(X)
            eps_x_list.append(ex)
            eps_y_list.append(ey)

        s_elements = np.arange(len(eps_x_list))

        fig, ax1 = plt.subplots(figsize=(8, 4.5))
        ax1.plot(s_elements, eps_x_list, label="εx", color="blue")
        ax1.set_xlabel("Number of elements")
        ax1.set_ylabel("εx (m·rad)", color="blue")
        ax1.tick_params(axis='y', labelcolor="blue")
        ax1.grid(True, linestyle=':')

        ax2 = ax1.twinx()
        ax2.plot(s_elements, eps_y_list, color="orange", label="εy")
        ax2.set_ylabel("εy (m·rad)", color="orange")
        ax2.tick_params(axis='y', labelcolor="orange")

        plt.title("Emittance Evolution (x vs y)")
        fig.tight_layout()
        plt.show()

# ==========================================
# 4. Tracking Processing Loops
# ==========================================

btn_track_beam = widgets.Button(description="Track 1000 Particles", button_style='success', icon='play', layout=widgets.Layout(width='100%'))
slider_lattice_element = widgets.IntSlider(value=0, min=0, max=100, step=1, description='Element:', continuous_update=False, layout=widgets.Layout(width='100%'))

def refresh_plots():
    plot_real_space(slider_lattice_element.value)
    plot_phase_space(slider_lattice_element.value)
    plot_beam_envelope()
    plot_emittance_evolution()

def run_beam_tracking():
    global beam_tracking_data
    n_particles = 100

    try:
        initial_particles = np.zeros((6, n_particles))
        initial_particles[0, :] = np.random.normal(0, 1, n_particles) * slider_x_rms.value * 1e-3
        initial_particles[1, :] = np.random.normal(0, 1, n_particles) * slider_xp_rms.value * 1e-3
        initial_particles[2, :] = np.random.normal(0, 1, n_particles) * slider_y_rms.value * 1e-3
        initial_particles[3, :] = np.random.normal(0, 1, n_particles) * slider_yp_rms.value * 1e-3
        initial_particles[4, :] = np.random.normal(0, 1, n_particles) * slider_dp_rms.value * 1e-3

        beam_tracking_data = at.lattice_pass(lattice, initial_particles, nturns=1, refpts=refpts)
        slider_lattice_element.max = beam_tracking_data.shape[2] - 1
        return True
    except Exception as e:
        with out_status:
            clear_output()
            print(f"Tracking error: {e}")
        return False

def handle_track_beam_click(b):
    with out_status:
        clear_output()
        while run_beam_tracking():
            slider_lattice_element.value = 0
            refresh_plots()
            print("Tracking complete!")

def handle_element_slider_change(change):
    if beam_tracking_data is not None:
        plot_real_space(slider_lattice_element.value)
        plot_phase_space(slider_lattice_element.value)

def handle_parameter_slider_change(change):
    if beam_tracking_data is not None:
        if run_beam_tracking():
            refresh_plots()

# Register observers targeting explicit value streams
slider_lattice_element.observe(handle_element_slider_change, names='value')
btn_track_beam.on_click(handle_track_beam_click)

for slider in [slider_x_rms, slider_xp_rms, slider_y_rms, slider_yp_rms, slider_dp_rms]:
    slider.observe(handle_parameter_slider_change, names='value')

# ==========================================
# 5. UI Layout Structural Construction
# ==========================================

panel_execution_control = widgets.VBox(
    [
        widgets.HTML("<h3>Execution</h3>"),
        btn_track_beam,
        out_status,
        widgets.Box(layout=widgets.Layout(height='20px')),
        widgets.HTML("<b>Select Lattice Element:</b>"),
        slider_lattice_element
    ],
    layout=widgets.Layout(width='100%', padding='15px', border='1px solid #ddd', border_radius='6px', background_color='#fcfcfc', grid_gap='10px')
)

panel_parameters_sidebar = widgets.VBox(
    [
        widgets.HTML("<h3>1000 Particles Distribution</h3>"),
        widgets.HTML("<b>Standard Deviations (RMS):</b>"),
        slider_x_rms, slider_xp_rms, slider_y_rms, slider_yp_rms, slider_dp_rms,
        widgets.Box(layout=widgets.Layout(height='10px')),
        widgets.HTML("<hr><b>OR Calculate from Emittance:</b>"),
        input_emit_x, input_emit_y,
        btn_calculate_from_emit,
        widgets.Box(layout=widgets.Layout(height='10px')),
        out_emittance_calc,
        panel_execution_control
    ],
    layout=widgets.Layout(width='320px', padding='15px', border='1px solid #ddd', border_radius='6px', background_color='#fdfbf9', grid_gap='5px')
)

tab_spaces_view = widgets.VBox([out_real_space, out_phase_space], layout=center_layout)
tab_envelope_view = widgets.VBox([out_envelope], layout=center_layout)
tab_emittance_view = widgets.VBox([out_emittance], layout=center_layout)

# Inject baseline descriptive info strings
with out_real_space: print("Run tracking on the left to see spaces.")
with out_envelope: print("Run tracking on the left to see beam envelope.")
with out_emittance: print("Run tracking on the left to see emittance data.")

results_tab_panel = widgets.Tab(layout=widgets.Layout(flex='1 1 auto'))
results_tab_panel.children = [tab_spaces_view, tab_envelope_view, tab_emittance_view]
results_tab_panel.set_title(0, 'Phase & Real Space')
results_tab_panel.set_title(1, 'Beam Envelope')
results_tab_panel.set_title(2, 'Emittance Growth')

main_interface_layout = widgets.HBox(
    [panel_parameters_sidebar, results_tab_panel],
    layout=widgets.Layout(width='100%', align_items='stretch', grid_gap='15px')
)

# Render Application Container
display(main_interface_layout)

In [ ]:
#@title Play With Parameters!

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import at

# ==========================================
# 1. Slider & Widget Controls Initialization
# ==========================================

qf_slider = widgets.FloatSlider(value=1.2, min=0.0, max=3.0, step=0.05, description='QF K:', continuous_update=False)
qd_slider = widgets.FloatSlider(value=-1.2, min=-3.0, max=0.0, step=0.05, description='QD K:', continuous_update=False)
sf_slider = widgets.FloatSlider(value=10.0, min=-50.0, max=50.0, step=0.5, description='SF H:', continuous_update=False)
sd_slider = widgets.FloatSlider(value=-12.0, min=-50.0, max=50.0, step=0.5, description='SD H:', continuous_update=False)
bend_slider = widgets.FloatSlider(value=0.3, min=0.01, max=1.0, step=0.01, description='B angle:', continuous_update=False)
ncell_slider = widgets.IntSlider(value=4, min=1, max=20, step=1, description='Cells:', continuous_update=False)
energy_slider = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Energy [GeV]:', continuous_update=False)

all_lattice_sliders = [qf_slider, qd_slider, sf_slider, sd_slider, bend_slider, ncell_slider, energy_slider]

x_range = widgets.IntRangeSlider(value=[0, 2], min=0, max=10, step=1, description='X Range:', continuous_update=False, layout=widgets.Layout(width='100%'))
y_range = widgets.IntRangeSlider(value=[0, 2], min=0, max=10, step=1, description='Y Range:', continuous_update=False, layout=widgets.Layout(width='100%'))
order_slider = widgets.IntSlider(value=2, min=1, max=5, step=1, description='Res. Order:', continuous_update=False, layout=widgets.Layout(width='100%'))

button_ts = widgets.Button(description="Force Refresh", icon='refresh', layout=widgets.Layout(width='100%'))
all_tune_sliders = [x_range, y_range, order_slider]

output_table = widgets.Output()
output_beta_plot = widgets.Output()
output_ts_plot = widgets.Output(layout=widgets.Layout(padding='10px', min_width='400px'))

# Global simulation tracking arrays
ring_sim = None
current_tunes = [0.0, 0.0]

# ==========================================
# 2. Physics Simulation & Plotting Logic
# ==========================================

def draw_tune_space(tunes_to_draw):
    """Plots the fractional working point and overlapping resonance diagram grid."""
    with output_ts_plot:
        clear_output(wait=True)
        if ring_sim is None:
            pass
            #print('No lattice defined. Please adjust parameters on the left first.')
            return

        try:
            plt.figure(figsize=(5.5, 5.5))
            x_limits = list(x_range.value)
            y_limits = list(y_range.value)

            # Draw base resonance line limits using internal utility helper
            tunespace(tunes_to_draw, x_limits, y_limits, order=order_slider.value)

            # Superimpose active working fractional tune indicator position on the canvas
            plt.plot(tunes_to_draw[0], tunes_to_draw[1], 'ro', markersize=8, label=f'Current Tune ({tunes_to_draw[0]:.3f}, {tunes_to_draw[1]:.3f})')
            plt.legend(loc='upper right')
            plt.title(f"Tune Space (Order {order_slider.value})")
            plt.show()
        except Exception as e:
          pass
            #print(f"Plotting error: {e}")

def update_lattice(change=None):
    """Rebuilds the AT Lattice definition on value updates and evaluates optics stability."""
    global ring_sim, current_tunes

    with output_table: clear_output(wait=True)
    with output_beta_plot: clear_output(wait=True)

    try:
        D = at.Drift('D', 0.5)
        QF = at.Quadrupole('QF', 0.3, qf_slider.value)
        QD = at.Quadrupole('QD', 0.3, qd_slider.value)
        B = at.Dipole('B', 1.0, bend_slider.value, 0.0)
        SF = at.Sextupole('SF', 0.1, sf_slider.value)
        SD = at.Sextupole('SD', 0.1, sd_slider.value)

        cell = [D, QF, D, B, D, QD, D, SF, D, SD, D]
        ring_list = cell * ncell_slider.value

        ring_sim = at.Lattice(ring_list, energy=energy_slider.value * 1e9, particle='electron')
        s = ring_sim.get_s_pos()

        # Calculate closed-orbit parameters
        _, tunes, chroms, _ = ring_sim.linopt(0.0, refpts=np.arange(len(ring_sim)+1), get_chrom=True, coupled=False)
        current_tunes = tunes

        with output_table:
            html_table = f"""
            <table style="width:100%; border-collapse: collapse; margin-top: 10px;">
                <tr style="background-color: #f2f2f2; border-bottom: 2px solid #ddd;">
                    <th style="padding: 8px; text-align: left;">Parameter</th>
                    <th style="padding: 8px; text-align: left;">Value</th>
                </tr>
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px;"><b>Circumference</b></td>
                    <td style="padding: 8px;">{s[-1]:.3f} m</td>
                </tr>
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px;"><b>Horizontal Tune (Qx)</b></td>
                    <td style="padding: 8px;">{tunes[0]:.4f}</td>
                </tr>
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px;"><b>Vertical Tune (Qy)</b></td>
                    <td style="padding: 8px;">{tunes[1]:.4f}</td>
                </tr>
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px;"><b>Horizontal Chromaticity (ξx)</b></td>
                    <td style="padding: 8px;">{chroms[0]:.4f}</td>
                </tr>
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px;"><b>Vertical Chromaticity (ξy)</b></td>
                    <td style="padding: 8px;">{chroms[1]:.4f}</td>
                </tr>
            </table>
            """
            display(widgets.HTML(html_table))

        with output_beta_plot:
            ring_sim.plot_beta()
            plt.show()

        draw_tune_space(current_tunes)

    except Exception as e:
        with output_table:
            pass
            #print("LATTICE ERROR / UNSTABLE OPTICS:")
            #print(e)

# ==========================================
# 3. Signals & Interface Layout Assembly
# ==========================================

# Bind core recalculation loops to parameter adjustments
for slider in all_lattice_sliders:
    slider.observe(update_lattice, names='value')

# Bind viewport refresh adjustments to view bounds sliders
def handle_tune_space_slider_change(change):
    draw_tune_space(current_tunes)

for slider in all_tune_sliders:
    slider.observe(handle_tune_space_slider_change, names='value')

button_ts.on_click(lambda b: draw_tune_space(current_tunes))

left_control_panel = widgets.VBox(
    [
        widgets.HTML("<h2>Ring Parameters</h2><hr>"),
        widgets.HTML("<b>Quadrupole Strengths:</b>"), qf_slider, qd_slider,
        widgets.HTML("<b>Sextupole Strengths:</b>"), sf_slider, sd_slider,
        widgets.HTML("<b>Geometry & Beam:</b>"), bend_slider, ncell_slider, energy_slider
    ],
    layout=widgets.Layout(
        width='320px', padding='15px', border='1px solid #ddd',
        border_radius='8px', background_color='#fafafa', grid_gap='8px'
    )
)

ts_settings_panel = widgets.VBox(
    [
        widgets.HTML("<b>Tune Space Limits</b><br>"),
        x_range, y_range, order_slider,
        widgets.Box(layout=widgets.Layout(height='15px')),
        button_ts
    ],
    layout=widgets.Layout(
        width='280px', min_width='240px', padding='12px', border='1px solid #eee',
        border_radius='6px', background_color='#fcfcfc', grid_gap='6px'
    )
)

tab_tune_space = widgets.HBox(
    [ts_settings_panel, output_ts_plot],
    layout=widgets.Layout(width='100%', align_items='stretch', padding='10px')
)

right_results_tabs = widgets.Tab(layout=widgets.Layout(flex='1 1 auto'))
right_results_tabs.children = [output_table, output_beta_plot, tab_tune_space]
right_results_tabs.set_title(0, 'Summary Table')
right_results_tabs.set_title(1, 'Beta Functions')
right_results_tabs.set_title(2, 'Tune Space Diagram')

interactive_app = widgets.HBox(
    [left_control_panel, right_results_tabs],
    layout=widgets.Layout(width='100%', align_items='stretch', grid_gap='20px')
)

# Initial execution call to populate interface
update_lattice()

# Render Application Component
display(interactive_app)

/usr/local/lib/python3.12/dist-packages/at/lattice/lattice_object.py:554: AtWarning: AT tracking still assumes beta==1
Make sure your particle is ultra-relativistic
  setattr(self, key, value)
